# Fermionic 3D toric code, L=2 OBC — architecture ladder, magnetic line ($h_z=0$)

Four tiers at $h_x \in \{0.1,0.2,0.3,0.5,0.7,1.0\}$, $h_z=0$: a sign-blind
GeoCNN control (`plain`), an approximately-symmetric gridinv trunk with no
head (`asymm`), and the same trunk with a **frozen** analytic GF(2) sign head
at flux-penalty $\kappa=0$ (targets the true $E_0$) and $\kappa=6$ (targets
the head's own flux-sector-projected floor). Referee: dense ED on the full
$2^{12}=4096$-dim Hilbert space (`exact_diag_fermionic_L2_OBC_hx*_hz0.0.json`).

Many `anaC_k0` runs (and two `asymm` runs) hit the known $\kappa=0$ variance
wall (BLOG 2026-08-20) and were stopped by the divergence guard; their banked
last-sane state is real data, marked with $\times$ in the learning-curve
panels rather than dropped.

## 1. Config + loaders

In [ ]:
# %% 1. CONFIG — knobs live here -------------------------------------------
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})

ROOT = Path("../../results/fermionic_hx_ladder")
FIGS = Path("../figs")

HX_VALUES = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
TIERS = ["plain", "asymm", "anaC_k0", "anaC_k6"]
TIER_LABEL = {
    "plain": "sign-blind GeoCNN (plain)",
    "asymm": "approx-symm gridinv (asymm, no head)",
    "anaC_k0": r"frozen analytic head, $\kappa$=0",
    "anaC_k6": r"frozen analytic head, $\kappa$=6",
}
# Okabe-Ito palette (plot-style-spec): plain gray, asymm blue, anaC_k0 orange, anaC_k6 green.
TIER_COLOR = {
    "plain": "#808080",
    "asymm": "#0072B2",
    "anaC_k0": "#E69F00",
    "anaC_k6": "#009E73",
}
RUN_PREFIX = {"plain": "geocnn", "asymm": "gridinv", "anaC_k0": "gridinv", "anaC_k6": "gridinv"}
RUN_SUFFIX = {"plain": "plain", "asymm": "k2_asymm", "anaC_k0": "k2_anaC_k0", "anaC_k6": "k2_anaC_k6"}


def run_name(hx, tier):
    return f"{RUN_PREFIX[tier]}_fermionic_L2_OBC_hx{hx}_hz0.0_{RUN_SUFFIX[tier]}"


def load_curve(name):
    d = json.load(open(ROOT / f"{name}.curve.json"))
    c = d["curve"]
    return np.asarray(c["step"], float), np.asarray(c["energy"], float)


def load_snapshots(name):
    return json.load(open(ROOT / f"{name}.snapshots.json"))["series"]


ED = {hx: json.load(open(ROOT / f"exact_diag_fermionic_L2_OBC_hx{hx}_hz0.0.json")) for hx in HX_VALUES}

# Built by analysis/scripts/hx_ladder_summary.py from the run + ED-referee JSONs.
SUMMARY = json.load(open(ROOT / "summary.json"))
SUMMARY_BY_KEY = {(r["hx"], r["tier"]): r for r in SUMMARY}

# Intrinsic full-space sign match of the FROZEN analytic head vs ED (head never
# trains, so this is fixed by hx alone), and the ED ground state's flux-sector
# purity min<u_c> (how much of the true GS sits outside the h=0 sector the head
# is exact in). Source: leader-task log, slurm tc-fhxladder-57836502_0.out.
HEAD_INTRINSIC_SIGN_MATCH = {0.1: 0.9975, 0.2: 0.9890, 0.3: 0.9710, 0.5: 0.9125, 0.7: 0.8560, 1.0: 0.7839}
HEAD_INTRINSIC_MIN_U = {0.1: 0.990, 0.2: 0.947, 0.3: 0.829, 0.5: 0.397, 0.7: 0.157, 1.0: 0.053}

print(f"loaded {len(SUMMARY)} runs across {len(HX_VALUES)} hx points x {len(TIERS)} tiers")


## 2. Headline table

In [ ]:
# %% 2. headline table -----------------------------------------------------
hdr = (f"{'hx':>5} {'tier':<10} {'E_final':>12} {'rel_err':>10} "
       f"{'fidelity':>9} {'sign_match':>11} {'Vscore':>10} {'div':>6} {'last_step':>9}")
print(hdr)
print("-" * len(hdr))
for hx in HX_VALUES:
    for tier in TIERS:
        r = SUMMARY_BY_KEY[(hx, tier)]
        print(f"{hx:>5} {tier:<10} {r['E']:>12.5f} {r['rel']:>10.2e} "
              f"{r['fidelity']:>9.4f} {r['sign_match']:>11.4f} {r['Vscore']:>10.3g} "
              f"{str(r['diverged']):>6} {r['last_step']:>9}")
    print()


## 3. Learning curves (2x3 grid, one panel per $h_x$)

In [ ]:
# %% 3. learning curves, 2x3 grid over hx -----------------------------------
fig, axes = plt.subplots(2, 3, figsize=(14.5, 8.2))
for ax, hx in zip(axes.flat, HX_VALUES):
    for tier in TIERS:
        name = run_name(hx, tier)
        step, E = load_curve(name)
        r = SUMMARY_BY_KEY[(hx, tier)]
        ax.plot(step, E, lw=1.4, alpha=0.9, color=TIER_COLOR[tier], label=TIER_LABEL[tier])
        if r["diverged"]:
            ax.plot(step[-1], E[-1], marker="x", color=TIER_COLOR[tier], ms=9, mew=2, zorder=5)
    ax.axhline(ED[hx]["E0"], color="k", lw=1.2, zorder=1, label="$E_0$ (ED)")
    ax.set_title(f"$h_x={hx}$")
    ax.set_xlabel("SR step")
    ax.set_ylabel(r"$\langle H \rangle$")

handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=5, frameon=False, bbox_to_anchor=(0.5, -0.02), fontsize=9)
fig.suptitle(
    "Fermionic L=2 OBC arch ladder — learning curves per $h_x$ "
    "($\times$ = last state before the divergence guard stopped the run)",
    y=1.02,
)
fig.tight_layout()
# plt.savefig(FIGS / "fermionic_hx_ladder.png", dpi=300, bbox_inches="tight")
plt.show()


## 4. Relative energy error vs $h_x$

In [ ]:
# %% 4. relative error vs hx, one line per tier + anaC-head intrinsic ref ----
fig, ax = plt.subplots(figsize=(7.5, 5.4))
for tier in TIERS:
    rel = [SUMMARY_BY_KEY[(hx, tier)]["rel"] for hx in HX_VALUES]
    ax.plot(HX_VALUES, rel, marker="o", ms=5.5, lw=1.7, color=TIER_COLOR[tier], label=TIER_LABEL[tier])

# Sign-structure cost alone, with no trunk/optimization error: 1 - (head's own
# intrinsic sign match). Not literally the same quantity as rel(E), but the
# floor the anaC tiers are fighting.
intrinsic = [1.0 - HEAD_INTRINSIC_SIGN_MATCH[hx] for hx in HX_VALUES]
ax.plot(HX_VALUES, intrinsic, marker="s", ms=5.5, lw=1.3, ls="--", color="k",
        label=r"anaC head intrinsic $1-$sign match")

ax.set_yscale("log")
ax.set_xlabel("$h_x$")
ax.set_ylabel(r"$|E_{final}-E_0|/|E_0|$")
ax.set_title("Relative energy error vs $h_x$ (magnetic line, $h_z=0$)")
ax.legend(frameon=False, fontsize=8.5)
fig.tight_layout()
# plt.savefig(FIGS / "fermionic_hx_ladder_relerr.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Fidelity and sign match vs $h_x$

In [ ]:
# %% 5. fidelity + sign match vs hx, side by side ----------------------------
fig, (ax_fid, ax_sm) = plt.subplots(1, 2, figsize=(12.8, 4.8))
for tier in TIERS:
    fid = [SUMMARY_BY_KEY[(hx, tier)]["fidelity"] for hx in HX_VALUES]
    sm = [SUMMARY_BY_KEY[(hx, tier)]["sign_match"] for hx in HX_VALUES]
    ax_fid.plot(HX_VALUES, fid, marker="o", ms=5.5, lw=1.7, color=TIER_COLOR[tier], label=TIER_LABEL[tier])
    ax_sm.plot(HX_VALUES, sm, marker="o", ms=5.5, lw=1.7, color=TIER_COLOR[tier], label=TIER_LABEL[tier])

intrinsic_sm = [HEAD_INTRINSIC_SIGN_MATCH[hx] for hx in HX_VALUES]
ax_sm.plot(HX_VALUES, intrinsic_sm, marker="s", ms=5.5, lw=1.3, ls="--", color="k",
           label="anaC head intrinsic sign match")

ax_fid.set_xlabel("$h_x$")
ax_fid.set_ylabel(r"fidelity $|\langle\psi_{NQS}|\psi_{ED}\rangle|^2$")
ax_fid.set_title("Fidelity vs $h_x$")
ax_fid.legend(frameon=False, fontsize=7.5, loc="lower right")

ax_sm.set_xlabel("$h_x$")
ax_sm.set_ylabel("weighted sign match")
ax_sm.set_title("Sign match vs $h_x$")
ax_sm.legend(frameon=False, fontsize=7.5, loc="lower left")

fig.tight_layout()
# plt.savefig(FIGS / "fermionic_hx_ladder_fid_sign.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. Reading

**Small $h_x$ (0.1–0.3): the frozen head wins by 1–2 orders of magnitude.**
`anaC_k0` has the lowest relative error at $h_x=0.1,0.2,0.3$ (1.2e-3, 4.2e-3,
1.0e-2) — the flux sector is nearly conserved ($\text{min}\langle u_c\rangle$
= 0.990, 0.947, 0.829), so the frozen analytic sign is close to exact and
training only has to fix the trunk's amplitudes. The sign-blind tiers
(`plain`, `asymm`) are still learning the sign structure from scratch and sit
at 5–13% relative error over the same range.

**Large $h_x$ (0.5–1.0): the sign-blind trunk overtakes and wins big.**
`asymm` reaches rel. err. $3\times10^{-3}$, $2\times10^{-7}$, $6\times10^{-11}$
at $h_x=0.5,0.7,1.0$ — the field polarizes the state, sign structure becomes
almost trivial (the sign-blind sign_match rises to 1.0 by $h_x=0.7$), and an
unconstrained ansatz has nothing left to fight. Meanwhile the frozen head's
approximation *degrades* monotonically with $h_x$ (intrinsic sign match
0.9975 → 0.7839) because flux sectors mix as $\sigma^x$ is turned on and the
head is only exact at $h_x=0$ — it is now fighting the wrong battle.

**$\kappa=0$ vs $\kappa=6$.** $\kappa=0$ (targets the true, unprojected
$E_0$) hits the known SR variance wall at *every* $h_x$ here and is stopped
early by the divergence guard (last_step 30–113 of 300) — but its banked
pre-divergence state is still the best `anaC` number at small $h_x$.
$\kappa=6$ (sector-projected floor) trains stably almost everywhere
(diverges only at $h_x=1.0$) and drives Vscore down to $10^{-9}$–$10^{-21}$ —
but that is convergence to its *own* restricted-sector target, not the true
GS: its relative error to true $E_0$ actually *grows* with $h_x$ (1.3e-3 →
22.9%), a reminder that Vscore alone is not a proxy for accuracy once the
head is projecting onto the wrong sector.